In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
claims = spark.table("healthcare_dev.bronze.claims")

In [0]:
claims.count()

63970

In [0]:
window = Window.partitionBy("claim_id").orderBy(col("event_time").desc())

In [0]:
claims_clean = claims.withColumn("rn",row_number().over(window)).filter("rn=1").drop("rn")

In [0]:
claims_clean.count()

60970

In [0]:
(
    claims_clean.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "healthcare_dev.silver.claims"
    )
)

In [0]:
claims_late = claims_clean.filter(
    col("is_late_simulated") == True
)

claims_valid = (
    claims_clean
    .filter(
        col("is_late_simulated") == False
    )
    .drop("is_late_simulated")
)

In [0]:
(
claims_late.write
.format("delta")
.mode("overwrite")
.saveAsTable(
"healthcare_dev.silver.claims_quarantine"
)
)

(
claims_valid.write
.format("delta")
.mode("overwrite")
.saveAsTable(
"healthcare_dev.silver.claims"
)
)

In [0]:
claims.select("event_time").show(5)

+-------------------+
|         event_time|
+-------------------+
|1965-11-15 13:07:41|
|1977-02-21 11:52:41|
|1987-12-21 11:37:41|
|2010-03-01 11:37:41|
|2010-12-07 12:35:41|
+-------------------+
only showing top 5 rows


In [0]:
claims.select(max("event_time")).show()

+-------------------+
|    max(event_time)|
+-------------------+
|2019-09-16 16:31:50|
+-------------------+



In [0]:
claims.columns

['claim_id',
 'patient_id',
 'provider_id',
 'claim_amount',
 'currency',
 'diagnosis_code',
 'claim_status',
 'claim_type',
 'use',
 'event_time',
 'billable_start',
 'billable_end',
 'source_file',
 'ingestion_timestamp',
 'batch_id',
 'source_system',
 'is_late_simulated']

In [0]:
from pyspark.sql.functions import *

claims_valid = (

claims_valid

.withColumn(
    "updated_at",
    col("ingestion_timestamp")
)

)

In [0]:
(
claims_valid.write
.format("delta")
.mode("overwrite")
.option(
    "overwriteSchema",
    "true"
)
.saveAsTable(
    "healthcare_dev.silver.claims"
)
)

In [0]:
claims_valid.columns

['claim_id',
 'patient_id',
 'provider_id',
 'claim_amount',
 'currency',
 'diagnosis_code',
 'claim_status',
 'claim_type',
 'use',
 'event_time',
 'billable_start',
 'billable_end',
 'source_file',
 'ingestion_timestamp',
 'batch_id',
 'source_system',
 'updated_at']

In [0]:
from pyspark.sql.functions import *

incoming_claims = (
    spark.table(
        "healthcare_dev.bronze.claims"
    )

    .withColumn(
        "updated_at",
        current_timestamp()
    )
)

In [0]:
spark.table(
"healthcare_dev.silver.claims"
).select(
"claim_id"
).limit(3).show(truncate=False)

+------------------------------------+
|claim_id                            |
+------------------------------------+
|00001843-fe30-4a9f-9b9e-e3002013554f|
|0001abba-75b2-4ee7-9087-fd1db77da3f9|
|00021f77-1dfe-44f4-a161-bbfe8f0d9d13|
+------------------------------------+



In [0]:
incoming_claims = (

incoming_claims

.withColumn(

    "claim_amount",

    when(
        col("claim_id").isin(
            "00001843-fe30-4a9f-9b9e-e3002013554f","0001abba-75b2-4ee7-9087-fd1db77da3f9","00021f77-1dfe-44f4-a161-bbfe8f0d9d13"
        ),

        col("claim_amount")+200

    )

    .otherwise(
        col("claim_amount")
    )

)
)

In [0]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("claim_id").orderBy(col("updated_at").desc())

In [0]:
incoming_claims_dedup = (

incoming_claims
.withColumn("rn",row_number().over(window_spec))
.filter("rn=1").drop("rn")
)

In [0]:
from delta.tables import DeltaTable

silver = DeltaTable.forName(
    spark,
    "healthcare_dev.silver.claims"
)

(
silver.alias("target")

.merge(

incoming_claims_dedup.alias("source"),

"""
target.claim_id=
source.claim_id
"""

)

.whenMatchedUpdate(
condition=
"""
source.updated_at >
target.updated_at
""",

set={

"claim_amount":
"source.claim_amount",

"updated_at":
"source.updated_at"

}
)

.whenNotMatchedInsertAll()

.execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]